CREATE THE TEST DATASTRUCTURE FOR 3 EDGES

In [4]:
import pandas as pd
import networkx as nx
import os
import pickle

# Load your dataset
data = pd.read_csv('./data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')  # Replace 'test_data.csv' with your actual file name

# Pulizia dei nomi delle colonne (rimuove spazi bianchi all'inizio e alla fine)
data.columns = data.columns.str.strip()

# Specify the features to keep
features_to_keep = [
     # Identificativi dei nodi e parametri di base
    'Source IP', 'Destination IP', 'Source Port', 'Destination Port', 'Protocol', 'Timestamp', 'Label',

    # 1. Network Communication Features (Intensità e flussi)
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets',

    # 2. Contextual Features (Dinamiche temporali e comportamentali)
    'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Idle Mean', 'Idle Min', 'Active Mean',

    # 3. Knowledge-Based Features (Protocolli e statistiche pacchetti)
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
    'SYN Flag Count', 'ACK Flag Count', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward'
]

# Keep only the specified features
#filtered_data = data[features_to_keep]

# Filtra il dataset mantenendo solo le colonne esistenti tra quelle specificate
available_features = [f for f in features_to_keep if f in data.columns]
filtered_data = data[available_features].copy()

# Convert 'Timestamp' to datetime
filtered_data['Timestamp'] = pd.to_datetime(filtered_data['Timestamp'], dayfirst=True)

# Order the data by 'Timestamp'
filtered_data = filtered_data.sort_values(by='Timestamp')

# Save the temporally ordered data to a new file
filtered_data.to_csv('filtered_test_3edge.csv', index=False)

print("Filtered and temporally ordered data saved to 'filtered_test_3edge.csv'.")
print(f"Features removed (not found in the CSV): {set(features_to_keep) - set(available_features)}")


Filtered and temporally ordered data saved to 'filtered_test_3edge.csv'.
Features removed (not found in the CSV): set()


In [5]:
#check for inside of csv (just for test, no need for run)

# Load the CSV file
file_path = "filtered_test_3edge.csv"  # Replace with your actual file path
df = pd.read_csv(file_path)

# Check if the label column contains '1'
label_column = 'Label'  # Replace with the actual label column name if different
if label_column in df.columns:
    label_distribution = df[label_column].value_counts()
    print("Label Distribution:")
    print(label_distribution)

    if 1 in label_distribution.index:
        print("The CSV contains label '1'.")
    else:
        print("The CSV does NOT contain label '1'.")
else:
    print(f"'{label_column}' column not found in the CSV.")

Label Distribution:
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64
The CSV does NOT contain label '1'.


CREATED HOURLY GRAPH WITH 3 EDGES FROM TEST DATASET

In [7]:
def create_test_graphs_edge_labels(df, output_dir):
    """
    Split the DataFrame into hourly slices and create graphs for each slice.
    Each edge gets a valid label (e.g., 0 or 1) read from the DataFrame.

    Parameters:
        df (pd.DataFrame): The input DataFrame with temporal data.
        output_dir (str): Directory to save the graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Mapping per risolvere l'errore delle etichette testuali (BENIGN/DDoS)
    label_mapping = {'BENIGN': 0, 'DDoS': 1}

    # Group the DataFrame into hourly slices using the datetime index.
    # (Assumes the DataFrame index is already a DateTimeIndex)
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]

    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        # Print value counts of the 'Label' column in this time-slice.
        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())

        # Create a MultiDiGraph for this time-slice.
        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Source IP']
            dst_ip = row['Destination IP']

            # Convert the label to an int (if missing or invalid, you can decide a fallback; here we assume it is valid)
            #try:
            #    label = int(row['Label'])
            #except Exception as e:
            #    print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
            #    continue

            # TRADUZIONE LABEL: Gestisce stringhe come 'BENIGN' o 'DDoS'
            raw_label = row['Label']
            label = label_mapping.get(raw_label, 0) # Default a 0 se ignoto

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            # Add nodes if not already present.
            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            # Add edges for different interactions.
            # 1. Network Edge
            G.add_edge(src_ip, dst_ip, key='network',
                       label=label,
                  #selected fetures
                       interaction='network_communication')

            # 2. Context Edge
            G.add_edge(src_ip, dst_ip, key='context',
                       label=label,
                      #selected fetures
                       interaction='context')

            # 3. Knowledge Edge
            G.add_edge(src_ip, dst_ip, key='knowledge',
                       label=label,
                        #selected fetures
                       interaction='knowledge')

        # Save the graph as a .gpickle file.
        #graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.gpickle")
        #nx.write_gpickle(G, graph_path)

        graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.pkl")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f)

        print(f"Test graph for hour {slice_index} saved to {graph_path}")

# Usage Example for graph creation
if __name__ == "__main__":
    # Read CSV and prepare DataFrame.
    df_test = pd.read_csv('filtered_test_3edge.csv')

    # Pulizia nomi colonne dagli spazi extra tipici di CIC-IDS2017
    df_test.columns = df_test.columns.str.strip()

    df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'])
    # Set Timestamp as index and sort (required for grouping by hour)
    df_test = df_test.set_index('Timestamp').sort_index()

    output_test_dir = "3ed_tes_h_graphs"
    create_test_graphs_edge_labels(df_test, output_test_dir)


Hour 0:
Label
BENIGN    28635
DDoS      23865
Name: count, dtype: int64
Test graph for hour 0 saved to 3ed_tes_h_graphs\test_graph_hour_0.pkl
Hour 1:
Label
DDoS      104162
BENIGN     67531
Name: count, dtype: int64
Test graph for hour 1 saved to 3ed_tes_h_graphs\test_graph_hour_1.pkl
Hour 2:
Label
BENIGN    1552
Name: count, dtype: int64
Test graph for hour 2 saved to 3ed_tes_h_graphs\test_graph_hour_2.pkl


In [9]:
#########Riplika of previous code:

def add_node_features(G):
    """
    Adds additional features to nodes in the graph, including:
    - Node degree
    - Community ID
    - Temporal activity (average edge count per node)
    - Node centrality (betweenness centrality)

    Parameters:
        G (nx.MultiDiGraph): The input graph.

    Returns:
        nx.MultiDiGraph: The graph with added node features.
    """
    # Add degree
    for node in G.nodes:
        G.nodes[node]['degree'] = G.degree[node]

    # Add community detection (Label Propagation)
    undirected_graph = nx.Graph(G)  # Convert to undirected for community detection
    communities = nx.community.label_propagation_communities(undirected_graph)
    community_mapping = {node: community_id for community_id, community in enumerate(communities) for node in community}

    for node in G.nodes:
        G.nodes[node]['community'] = community_mapping.get(node, -1)

    # Add centrality (Betweenness Centrality)
    centrality = nx.betweenness_centrality(G)
    for node, value in centrality.items():
        G.nodes[node]['centrality'] = value

    return G

def create_test_graphs_edge_labels(df, output_dir):
    """
    Split the DataFrame into hourly slices and create graphs for each slice.
    Each edge gets a valid label (e.g., 0 or 1) read from the DataFrame.

    Parameters:
        df (pd.DataFrame): The input DataFrame with temporal data.
        output_dir (str): Directory to save the graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    label_mapping = {'BENIGN': 0, 'DDoS': 1}

    # Group the DataFrame into hourly slices using the datetime index.
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]

    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        # Print value counts of the 'Label' column in this time-slice.
        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())

        # Create a MultiDiGraph for this time-slice.
        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Source IP']
            dst_ip = row['Destination IP']

            # TRADUZIONE LABEL: Trasforma la stringa in intero (0 o 1) [cite: 470]
            label = label_mapping.get(row['Label'], 0)

            # Convert the label to an int (if missing or invalid, you can decide a fallback; here we assume it is valid)
            #try:
            #    label = int(row['Label'])
            #except Exception as e:
            #    print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
            #    continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            # Add nodes if not already present.
            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            # Add edges for different interactions.
            # 1. Network Edge
            G.add_edge(src_ip, dst_ip, key='network',
                       label=label,
                       #selected fetures
                       interaction='network_communication')

            # 2. Context Edge
            G.add_edge(src_ip, dst_ip, key='context',
                       label=label,
                       #selected fetures
                       interaction='context')

            # 3. Knowledge Edge
            G.add_edge(src_ip, dst_ip, key='knowledge',
                       label=label,
                       #selected fetures
                       interaction='knowledge')

        # Add node features
        G = add_node_features(G)

        # Save the graph as a .gpickle file.
        #graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.gpickle")
        #nx.write_gpickle(G, graph_path)

        graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.pkl")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f)
        print(f"Test graph for hour {slice_index} saved to {graph_path}")

# Usage Example for graph creation
if __name__ == "__main__":
    # Read CSV and prepare DataFrame.
    df_test = pd.read_csv('filtered_test_3edge.csv')

    # Pulizia nomi colonne da spazi extra (comuni nel dataset CIC-IDS2017) [cite: 505]
    df_test.columns = df_test.columns.str.strip()

    df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'])
    # Set Timestamp as index and sort (required for grouping by hour)
    df_test = df_test.set_index('Timestamp').sort_index()

    output_test_dir = "3ed_tes_h_graphs"
    create_test_graphs_edge_labels(df_test, output_test_dir)

Hour 0:
Label
BENIGN    28635
DDoS      23865
Name: count, dtype: int64
Test graph for hour 0 saved to 3ed_tes_h_graphs\test_graph_hour_0.pkl
Hour 1:
Label
DDoS      104162
BENIGN     67531
Name: count, dtype: int64
Test graph for hour 1 saved to 3ed_tes_h_graphs\test_graph_hour_1.pkl
Hour 2:
Label
BENIGN    1552
Name: count, dtype: int64
Test graph for hour 2 saved to 3ed_tes_h_graphs\test_graph_hour_2.pkl


COMMUNITY DETECTION FOR GRAPHS AND THEN UPDATE THE GRAPH WITH THE LABEL OF COMMUNITY FOR EACH NODE

In [10]:

def detect_and_label_communities_lpa(graph):
    """
    Perform community detection using the Label Propagation Algorithm (LPA) and label nodes with community IDs.
    Adds 'x' attribute based on the 'community' label.

    Parameters:
        graph (nx.MultiDiGraph): Input graph.

    Returns:
        graph (nx.MultiDiGraph): Updated graph with community labels and 'x' attributes.
    """
    # Convert MultiDiGraph to Graph (undirected graph for LPA)
    undirected_graph = nx.Graph(graph)

    # Perform community detection using LPA
    communities = nx.community.label_propagation_communities(undirected_graph)

    # Assign community labels to nodes and add 'x' attribute
    for community_id, community in enumerate(communities):
        for node in community:
            graph.nodes[node]['community'] = community_id
            graph.nodes[node]['x'] = [community_id]  # 'x' is a feature; wrap in a list for PyTorch Geometric compatibility

    return graph


def process_graphs_with_lpa(input_dir, output_dir):
    """
    Detect communities using LPA, update graphs with community labels, and add 'x' attribute.

    Parameters:
        input_dir (str): Directory containing input graphs.
        output_dir (str): Directory to save updated graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Process each graph file in the input directory
    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue

        # Load the graph
        graph_path = os.path.join(input_dir, graph_file)
        G = nx.read_gpickle(graph_path)

        # Detect communities using LPA and label nodes
        G = detect_and_label_communities_lpa(G)

        # Save the updated graph
        updated_graph_path = os.path.join(output_dir, graph_file)
        nx.write_gpickle(G, updated_graph_path)
        print(f"Updated graph with LPA communities and 'x' attribute saved to {updated_graph_path}")


# Example usage
if __name__ == "__main__":
    # Input directory containing graphs
    input_graph_dir = "3ed_tes_h_graphs"

    # Output directory for updated graphs
    output_graph_dir = "3ed_tes_h_graphs_commun"

    # Process graphs and add community labels using LPA
    process_graphs_with_lpa(input_graph_dir, output_graph_dir)


CONVERT MULTIGRAPH TO HETRODATA

In [12]:
import torch
from torch_geometric.data import HeteroData

def multiDiGraph_to_hetero_with_label(G: nx.MultiDiGraph) -> HeteroData:
    """
    Converts a MultiDiGraph with multiple edge types to a HeteroData object.
    Preserves the 'label' field in data[rel_type].edge_label.
    """
    data = HeteroData()
    node_mapping = {node: i for i, node in enumerate(G.nodes())}
    data['ip'].num_nodes = G.number_of_nodes()

    # Add node-level features
    x = []
    community_labels = []
    for node in G.nodes():
        community = G.nodes[node].get('community', -1)
        community_labels.append(community)
        x.append([community])
    data['ip'].community = torch.tensor(community_labels, dtype=torch.long)
    data['ip'].x = torch.tensor(x, dtype=torch.float)

    # Process each edge from G.
    for u, v, key, edge_attrs in G.edges(data=True, keys=True):
        src = node_mapping[u]
        dst = node_mapping[v]
        rel_type = ('ip', key, 'ip')

        if rel_type not in data.edge_types:
            data[rel_type].edge_index = []
            data[rel_type].edge_attr = []
            data[rel_type].edge_label = []  # Container for the label

        data[rel_type].edge_index.append([src, dst])

        feature_vec = []
        if key == 'network':
            #for attr_name in [ #selected fetures]:
                #feature_vec.append(edge_attrs.get(attr_name, 0))
            feature_vec = [edge_attrs.get('Source Port', 0), edge_attrs.get('Destination Port', 0)]
        elif key == 'context': #selected fetures]:
                #feature_vec.append(edge_attrs.get(attr_name, 0))
            feature_vec = [edge_attrs.get('Flow Duration', 0), edge_attrs.get('Flow IAT Mean', 0)]
        elif key == 'knowledge':
            #for attr_name in [ #selected fetures]:
                #feature_vec.append(edge_attrs.get(attr_name, 0))
            feature_vec = [edge_attrs.get('Packet Length Std', 0), edge_attrs.get('SYN Flag Count', 0)]

        data[rel_type].edge_attr.append(feature_vec)
        # Save the label—this should now be valid (0 or 1)
        data[rel_type].edge_label.append(edge_attrs.get('label', -1))

    # Convert lists to tensors.
    for rel_type in data.edge_types:
        data[rel_type].edge_index = torch.tensor(data[rel_type].edge_index, dtype=torch.long).t().contiguous()
        if data[rel_type].edge_attr:
            data[rel_type].edge_attr = torch.tensor(data[rel_type].edge_attr, dtype=torch.float)
        if data[rel_type].edge_label:
            data[rel_type].edge_label = torch.tensor(data[rel_type].edge_label, dtype=torch.long)
    return data

def process_and_save_hetero_graphs_with_label(input_dir, output_dir):
    """
    Converts all .gpickle graphs in a directory to HeteroData objects and saves them as .pt,
    preserving the 'label' field in data[rel_type].edge_label.
    """
    os.makedirs(output_dir, exist_ok=True)
    if not os.path.exists(input_dir):
        print(f"Error: the input directory {input_dir} not exists!")
        return

    for graph_file in os.listdir(input_dir):
        #if not graph_file.endswith('.gpickle'):
        if not graph_file.endswith('.pkl'):
            continue

        graph_path = os.path.join(input_dir, graph_file)
        with open(graph_path, 'rb') as f:
            G = pickle.load(f)

        #G = nx.read_gpickle(graph_path)
        hetero_data = multiDiGraph_to_hetero_with_label(G)
        hetero_path = os.path.join(output_dir, graph_file.replace('.pkl', '.pt'))
        torch.save(hetero_data, hetero_path)
        print(f"Saved HeteroData with labels to {hetero_path}")

if __name__ == "__main__":
    input_test_dir = "3ed_tes_h_graphs_commun"         # Input .gpickle files (with communities added)
    output_test_pt_dir = "./3ed_tes_h_graphs_hetero_graphs" # Output .pt files
    process_and_save_hetero_graphs_with_label(input_test_dir, output_test_pt_dir)
